In [ ]:
#| default_exp core.parametrize

In [ ]:
#| export
from __future__ import annotations

import torch
import torch.nn as nn
from torch.nn.utils import parametrize

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

`torch.nn.utils.parametrize` lets a module *compute* its weight instead of storing it: the parameter that
is trained becomes `m.parametrizations.weight.original` — the **master** — and `m.weight` is whatever a
small module returns when given that master.
[`FakeQuantizeCallback`](../quantize/fake_quantize_callback.html) uses this to train through a rounded
weight while the optimizer keeps updating a floating-point one.

Everything else in fasterai reads and writes weights, so it has to know which of the two it is touching:

| | reading `m.weight` | reading `_master(m)` |
|---|---|---|
| a plain module | the parameter | the same parameter |
| a parametrized module | the computed weight, rebuilt on every access | the parameter being trained |

Writing is the sharper edge: `m.weight.copy_(x)` on a parametrized module lands on a tensor that is
thrown away as soon as the next forward recomputes it, and no error is raised. `_master(m).copy_(x)`
writes where it will be read from.

A parametrization also *adds modules*: `m.parametrizations` is a `ModuleDict` holding one
`ParametrizationList` per parametrized tensor, and both appear in `model.modules()`, immediately after
the module they belong to. `_plain_modules` walks a model without them — which is what any loop reading
`model.modules()` by position needs, such as `Sparsifier` pairing a convolution with the module
registered after it to find its BatchNorm.

In [ ]:
#| export
def _is_parametrized(m: nn.Module) -> bool:
    "Whether something computes `m.weight` from a master parameter"
    return parametrize.is_parametrized(m, 'weight')


def _master(m: nn.Module):
    "The parameter `m.weight` is computed from, or `m.weight` itself when nothing parametrizes it"
    return m.parametrizations.weight.original if _is_parametrized(m) else m.weight


def _plain_modules(
    model: nn.Module,
):
    "The modules of `model`, without the containers a parametrization inserts under the one it rewrites"
    for name, m in model.named_modules():
        if 'parametrizations' not in name.split('.'): yield m


def _unparametrize(
    m: nn.Module,
    leave_parametrized: bool = False,  # Keep the computed weight; False restores the master
) -> None:
    "Drop the weight parametrization, if there is one, and make `m.weight` an ordinary parameter again"
    if not _is_parametrized(m): return
    # a module parametrizing a second tensor is left to torch, whose removal deletes the property from
    # the injected class: a deepcopy of THAT module breaks, exactly as it would without this helper
    if len(m.parametrizations) > 1:
        parametrize.remove_parametrizations(m, 'weight', leave_parametrized)
        return
    master = m.parametrizations.weight.original
    if leave_parametrized:
        with torch.no_grad(): master.set_(m.weight.detach())  # the same object, so no optimizer rebinding
    del m._modules['parametrizations']
    # torch's own removal deletes the property from the injected class, which a deepcopy of this module
    # SHARES: giving the plain class back instead leaves every copy of it working
    m.__class__ = m.__class__.__bases__[0]
    if isinstance(master, nn.Parameter): m.register_parameter('weight', master)
    else: m.register_buffer('weight', master)

In [ ]:
show_doc(_is_parametrized)

In [ ]:
show_doc(_master)

In [ ]:
show_doc(_plain_modules)

In [ ]:
show_doc(_unparametrize)

---

## Usage

```python
from fasterai.core.parametrize import _master, _is_parametrized, _plain_modules, _unparametrize

_master(conv)                              # the parameter to score, mask, snapshot or rewind
_master(conv).data.mul_(mask)              # a write that survives the next forward
list(_plain_modules(model))                # the model's own modules, in registration order
_unparametrize(conv, leave_parametrized=True)   # bake the computed weight in
_unparametrize(conv)                            # drop the rounding, keep the master
```

`remove_parametrizations` keeps the master's identity: the object an optimizer holds before the
parametrization is registered is the same object it holds after one is removed, so nothing needs to be
rebound around either operation.

These helpers are private: they are how fasterai's own classes stay correct on a parametrized model, not
a public API. `_master` is what [`Sparsifier`](../sparse/sparsifier.html) and
[`Criteria`](criteria.html) call before touching a weight.

---

## See Also

- [FakeQuantizeCallback](../quantize/fake_quantize_callback.html) - Trains through a parametrized weight
- [Sparsifier](../sparse/sparsifier.html) - Masks, snapshots and rewinds the master
- [Criteria](criteria.html) - Scores the master, not the weight computed from it
- [Pruner](../prune/pruner.html) - Refuses a parametrized model, because it rewrites what it traces

Tests live in `nbs/tests/test_parametrize.ipynb`.